# Nishimoto and Gallant (2011) on CRCNS mt-2

Paper: Nishimoto S, Gallant JL (2011), "A Three-Dimensional Spatiotemporal Receptive Field Model
Explains Responses of Area MT Neurons to Naturalistic Movies", J Neurosci 31(41):14551-14564,
doi:10.1523/jneurosci.6801-10.2011.

Dataset: CRCNS mt-2, "Extracellular recordings from area MT of awake macaques in response to
naturalistic movies", doi:10.6080/K0DN4374, read from the public bucket
`crcnsarchive/mt-2/live/`. CRCNS data sets are not versioned; the version is recorded as
`unversioned` and the next cell prints the MD5 sums of `checksums.md5` for the 45 files read, so a
later change can be detected. The `live` folder is the only copy in the bucket (there is no
`old` folder), so it is the version pinned.

Claims: `claims/nishimoto-2011.yaml`, written from the version of record (main text, Methods and
legends; the paper has no supplement beyond a movie of stimulus spectra and links no code).

The model cannot be fitted here at the paper's size (about 6000 filters at 11 delays, boosted
five times, for 12 models and 45 neurons) within the time limit. The filter bank follows the
paper's rules (about 7000 complex and 1100 simple channels), and the regression weights are
constrained to be pooled over space onto 763 regressors (see `mt2.py`). Every difference from
the paper is listed in the summary and in the attempt record.

| Claim | Tested here |
|---|---|
| fig3b-mean-prediction | tested |
| fig3b-explainable-variance | blocked: data_missing (no single repeats) |
| fig3a-example-neuron | tested |
| fig2b-contrast-normalization-no-effect | tested |
| fig2b-compressive-improves | tested |
| fig2b-expansive-decreases | tested |
| fig2b-divisive-improves | tested |
| fig2b-compressive-divisive-best | tested |
| fig2b-no-interaction | tested |
| complex-cells-dominate | tested |
| fig6a-null-on-plane | tested |
| fig6a-excitatory-on-plane | tested |
| fig6b-suppressive-off-plane | tested |
| fig7a-partial-ring | tested |
| fig7a-no-corr-prediction-hv | tested |
| fig7b-pattern-index | tested |
| fig7c-hv-pattern-correlation | tested |
| fig7c-no-corr-prediction-pattern | tested |
| natural-movie-control | tested |
| fig8-recovery-simulated | tested |

In [ ]:
import os, time, resource
os.environ.setdefault("OMP_NUM_THREADS", "2")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import wilcoxon, ranksums, pearsonr
import mt2

T0 = time.time()
sizes = mt2.list_sizes()
names = sorted(sizes)
md5 = mt2.checksums()
print(len(names), "files;", sum(sizes.values()) / 1e9, "GB")
for n in names:
    print(n, md5[n], sizes[n])

## Fitting every neuron

Each neuron's file is streamed, its frames reduced to 32 x 32 pixels, the filter bank applied
with and without luminance and contrast normalization, and the 12 models of Figure 2 fitted by
boosting with early stopping (`mt2.process_neuron`). For the compressive and divisive model it
also fits the readings the claims file names (delays 1 to 10, folds as random partitions,
sigma chosen on held-out data, the 12 ms Gaussian as a full width), the complex-only and
simple-only models, the null model neurons of Figure 6 and the simulated neurons of Figure 8,
and drives the fitted model with gratings and plaids. Two neurons run at a time.

Null model neurons (fig6a-null-on-plane): 150 neurons, the k-th on the movies of neuron k mod 45
(movie reading (a)), each under the four combinations of the rate reading (rectified (a),
offset (b)) and the delay reading (all delays (a), one delay (b)); and 150 more on the movies of
one neuron, `ct0001` (movie reading (b)), with readings (a) and (a) only, for time.

In [ ]:
COMBOS = [("a", "a"), ("a", "b"), ("b", "a"), ("b", "b")]
SIM = "ct0001_arg0466d_128.mat"
specs = {n: [] for n in names}
for i in range(150):
    for ra, da in COMBOS:
        specs[names[i % len(names)]].append((("a", i), ra, da))
N_RUN = int(os.environ.get("MT2_N", len(names)))  # a subset while developing
for i in range(150 if N_RUN >= len(names) else 10):
    specs[SIM].append((("b", i), "a", "a"))
order = [SIM] + [n for n in sorted(names, key=lambda n: -sizes[n]) if n != SIM]
if N_RUN < len(names):
    order = [SIM] + [n for n in sorted(names, key=lambda n: sizes[n]) if n != SIM][:N_RUN - 1]
t = time.time()
with ProcessPoolExecutor(2) as ex:
    futs = {n: ex.submit(mt2.process_neuron, n, specs[n], n == SIM, 10 * names.index(n), 2) for n in order}
    R = {n: f.result() for n, f in futs.items()}
names_run = [n for n in names if n in R]
print(f"{len(R)} neurons in {(time.time() - t) / 60:.1f} min; per neuron "
      f"{np.median([R[n]['t_total'] for n in R]):.0f} s median, loading {np.median([R[n]['t_load'] for n in R]):.0f} s")
print("peak memory of a worker:", resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 1e6, "GB")

## Data check

In [ ]:
rows = []
for n in names_run:
    o = R[n]
    rows.append(dict(file=n[:6], cellid=o["cellid"], monkey=o["cellid"][0], frames=o["n_frames"],
                     crange=o["n_crange"], train_rows=o["n_train"], nan_in_crange=o["nan_train"],
                     val_sets="+".join(f"{k}:{v}" for k, v in o["sets"].items()),
                     reps_enh=o["repeats"].get("enhanced"), reps_nat=o["repeats"].get("natural"),
                     mean_count=round(o["mean_count"], 3),
                     motion_train=round(o["motion"]["train"], 3), motion_enh=round(o["motion"]["enhanced"], 3),
                     motion_nat=round(o["motion"].get("natural", np.nan), 3),
                     stim_min=o["stim_range"][0], stim_max=o["stim_range"][1], offset=o["offset"]))
D = pd.DataFrame(rows)
pd.set_option("display.width", 250, "display.max_columns", 30, "display.max_rows", 60)
print(D.to_string())
print("\nneurons by monkey:", D.monkey.value_counts().to_dict(), "; unique cell ids:", D.cellid.nunique())
print("training frames (crange):", D.crange.min(), "to", D.crange.max(), "mean", round(D.crange.mean()))
print("total frames per file:", sorted(D.frames.unique()))
print("files with a natural-movie validation set (nrange):", (D.motion_nat.notna()).sum())
print("validation repeats (largest per-frame denominator) enhanced:", D.reps_enh.min(), "to", D.reps_enh.max(),
      "; natural:", D.reps_nat.min(), "to", D.reps_nat.max())
print("frames with responses removed inside crange: median", D.nan_in_crange.median(), "range",
      D.nan_in_crange.min(), "to", D.nan_in_crange.max())
print("training-frame count denominators seen (repeats of training frames):",
      sorted({d for n in names_run for d in R[n]["train_denominators"]}))
nat = D[D.motion_nat.notna()]
print(f"\nmotion (mean absolute frame difference / mean contrast): training {D.motion_train.median():.3f}, "
      f"enhanced validation {D.motion_enh.median():.3f}, nrange {nat.motion_nat.median():.3f} (medians); "
      f"nrange below the other validation half in {(nat.motion_nat < nat.motion_enh).sum()} of {len(nat)}")

In [ ]:
# Recordings deposited twice: identical validation responses or identical training movies
V = {n: R[n]["psth"][-2000:] for n in names_run}
dup = []
for i, a in enumerate(names_run):
    for b in names_run[i + 1:]:
        if len(R[a]["psth"]) == len(R[b]["psth"]):
            c = np.corrcoef(V[a], V[b])[0, 1]
            if c > 0.9:
                dup.append((a[:6], b[:6], round(c, 3)))
print("pairs of neurons with validation responses correlated above 0.9:", dup if dup else "none")
same_resp = [(a[:6], b[:6]) for i, a in enumerate(names_run) for b in names_run[i + 1:]
             if len(R[a]["psth"]) == len(R[b]["psth"]) and np.allclose(np.nan_to_num(R[a]["psth"]), np.nan_to_num(R[b]["psth"]))]
print("pairs with identical response arrays:", same_resp if same_resp else "none")

In [ ]:
# Clock: the prediction of the compressive and divisive model against the validation response
# shifted in time (the response moved k frames later is compared with the prediction).
sh = R[names_run[0]]["fits"][mt2.MAIN]["shift_r"][0]
S = np.array([R[n]["fits"][mt2.MAIN]["shift_r"][1] for n in names_run])
peak = sh[np.argmax(S, 1)]
print("shift of the peak correlation (frames): counts", dict(zip(*np.unique(peak, return_counts=True))))
print(f"mean r at shift 0: {S[:, sh == 0].mean():.3f}; at |shift| >= 20 frames: {S[:, np.abs(sh) >= 20].mean():.3f}")
off = np.array([R[n]["offset"] for n in names_run])
print("delay-window offset chosen on held-out training data (frames earlier):", dict(zip(*np.unique(off, return_counts=True))))
dw = np.array([np.abs(R[n]["fits"][mt2.MAIN]["w"].reshape(mt2.DELAYS, -1)).sum(1) for n in names_run])
dw = dw / dw.sum(1, keepdims=True)
print("share of absolute weight per delay (median over neurons), delays 0..10:", np.round(np.median(dw, 0), 3))
oc = pd.DataFrame({n[:6]: {o: round(np.mean(list(v[1].values())), 3) for o, v in R[n]["offsets"].items()} for n in names_run}).T
print("validation r of the compressive and divisive model under each offset: mean", oc.mean().round(3).to_dict())
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(sh, S.T, color="0.7", lw=0.5)
ax.plot(sh, S.mean(0), "k", lw=2)
ax.set_xlabel("response shift (frames)"); ax.set_ylabel("r")
print("plotted r range:", S.min().round(3), "to", S.max().round(3))

## Per-neuron prediction accuracy

Following the paper (Results, Control to identify any bias, paragraph 2: "the predictions
reported throughout this manuscript reflect the average prediction for both natural and
motion-enhanced and natural movies (when the latter were available)"), a neuron's accuracy is the
mean of its two validation correlations where it has both sets, and the motion-enhanced
correlation otherwise. Which part of `vrange` is natural is read from `nrange` (see the data check).

In [ ]:
def acc(r):
    return float(np.mean(list(r.values())))

models = mt2.MODELS
IDX = [n[:6] for n in names_run]
A = {m: pd.Series([acc(R[n]["fits"][m]["r"]) for n in names_run], index=IDX) for m in models}
A_enh = {m: pd.Series([R[n]["fits"][m]["r"]["enhanced"] for n in names_run], index=IDX) for m in models}
monkey = np.array([R[n]["cellid"][0] for n in names_run])
def label(m):
    lum, a, div = m
    return f"{'lum ' if lum else ''}{ {1.0: 'linear', 0.5: 'compressive', 2.0: 'expansive'}[a]}{' +div' if div else ''}"
print(pd.DataFrame({"mean r": [A[m].mean() for m in models], "mean r (enhanced only)": [A_enh[m].mean() for m in models]},
                   index=[label(m) for m in models]).round(3))

## fig3b-mean-prediction

> The compressive and divisive V1 filter model predicts the validation responses of the 52 MT
> neurons with an average correlation of r = 0.52. (Expected 0.52 +- 0.052, over the 45 neurons
> in the dataset.)

In [ ]:
main = A[mt2.MAIN]
print(f"mean r, compressive and divisive model, {len(main)} neurons: {main.mean():.3f} (median {main.median():.3f}, "
      f"range {main.min():.3f} to {main.max():.3f})")
print(f"within 0.468 to 0.572: {0.468 <= main.mean() <= 0.572}")
print(f"motion-enhanced validation only: {A_enh[mt2.MAIN].mean():.3f}")
for mk in np.unique(monkey):
    print(f"monkey {mk}: n = {(monkey == mk).sum()}, mean r {main[monkey == mk].mean():.3f}")
rd_ = {}
rd_["delays 1 to 10 (b)"] = [acc(R[n]["fits"][mt2.MAIN]["delays_b"]["r"]) for n in names_run]
rd_["Gaussian 12 ms as FWHM (b)"] = [acc(R[n]["fits"][mt2.MAIN]["r_smooth_b"]) for n in names_run]
rd_["folds as random partitions (b)"] = [acc(R[n]["fits"][mt2.MAIN]["folds_b"]["r"]) for n in names_run]
rd_["sigma chosen on held-out data (a)"] = [acc(min(R[n]["sigma_a"].values(), key=lambda v: v[0])[1]) for n in names_run]
rd_["delay offset 0 for every neuron"] = [acc(R[n]["offsets"][0][1]) for n in names_run]
for k, v in rd_.items():
    print(f"reading {k}: mean r {np.mean(v):.3f}")
sig_choice = [min(R[n]["sigma_a"], key=lambda s: R[n]["sigma_a"][s][0]) for n in names_run]
print("sigma chosen (fraction of mean pooled response):", dict(zip(*np.unique(sig_choice, return_counts=True))))
print(f"neurons between 0.2 and 0.8 (Figure 3B span): {((main >= 0.2) & (main <= 0.8)).sum()} of {len(main)}; "
      f"below 0.2: {(main < 0.2).sum()}, above 0.8: {(main > 0.8).sum()}")
fig, ax = plt.subplots(figsize=(4, 3))
ax.hist(main, bins=np.arange(-0.1, 1.01, 0.05), color="0.4")
ax.set_xlabel("prediction correlation"); ax.set_ylabel("neurons")

**Result: not reproduced.** The compressive and divisive model predicts the 45 neurons with a mean correlation of 0.448 (median 0.454, range 0.221 to 0.730), below the tolerance band of 0.468 to 0.572. No reading reaches it: delays 1 to 10 give 0.449, folds as random partitions 0.446, sigma chosen on held-out data 0.451, offset 0 for every neuron 0.448, and the 12 ms Gaussian read as a full width 0.380. All 45 neurons lie between 0.2 and 0.8, as in Figure 3B. The model here is a spatially pooled version of the paper's (763 regressors), which may account for part of the gap.

## fig3b-explainable-variance

Blocked: the files hold only the response averaged over repeats (`psths`), so the noise ceiling
of David and Gallant (2005), which needs the single repeats, cannot be estimated. The data check
shows the repeat counts can be inferred (per-frame denominators) but not the trial responses.

## fig3a-example-neuron

> For one example neuron the correlation between model predictions and observed responses is
> r = 0.72 (expected 0.72 +- 0.072; the paper's neuron had 10 validation repeats).

In [ ]:
ex = A[mt2.MAIN]
hit = ex[(ex >= 0.648) & (ex <= 0.792)]
reps = {n[:6]: R[n]["repeats"] for n in names_run}
print(f"neurons with r in 0.648 to 0.792: {len(hit)}")
for k, v in hit.items():
    print(f"  {k}  r = {v:.3f}  repeat estimate {reps[k]}")
print("highest r:", ex.sort_values().tail(5).round(3).to_dict())
print("neurons whose motion-enhanced validation set shows 10 repeats:", [k for k, v in reps.items() if v.get("enhanced") == 10])

**Result: reproduced.** Three neurons have a prediction correlation within 0.648 to 0.792 (ct0013 0.710, ct0021 0.708, ct0039 0.730). None of them shows 10 validation repeats by the per-frame denominators (20 to 21); the two neurons that show 10 repeats (ct0028, ct0030) are not among them, so the paper's example neuron cannot be identified.

## Figure 2B: the 12 models

Each model's change of predictive power relative to the simplest model (linear, no luminance or
divisive normalization) is computed under the three readings of the claims file: (a) the percent
change of the mean correlation; (b) the same with the mean squared correlation; (c) the mean over
neurons of each neuron's percent change. Significance: Wilcoxon signed-rank test across neurons
on the per-neuron correlations, Bonferroni corrected over 11 comparisons (5 as a variant).

In [ ]:
base = (False, 1.0, False)
def change(m, Atab=A):
    a, b = Atab[m], Atab[base]
    pa = 100 * (a.mean() / b.mean() - 1)
    pb = 100 * ((a**2).mean() / (b**2).mean() - 1)
    pc = 100 * np.mean((a - b) / np.abs(b))
    p = wilcoxon(a, b).pvalue if m != base else np.nan
    return pa, pb, pc, p
rows = []
for m in models:
    pa, pb, pc, p = change(m)
    rows.append(dict(model=label(m), mean_r=A[m].mean(), pct_a=pa, pct_b=pb, pct_c=pc,
                     n_better=int((A[m] > A[base]).sum()), p=p, p_bonf11=min(1, p * 11) if m != base else np.nan,
                     p_bonf5=min(1, p * 5) if m != base else np.nan))
T2 = pd.DataFrame(rows).set_index("model")
print(T2.round(4).to_string())
print("\nreading (c) is sensitive to neurons with small baseline r: smallest baseline r", A[base].min().round(3),
      "; reading (c) without neurons whose baseline r < 0.1:",
      {label(m): round(100 * np.mean(((A[m] - A[base]) / A[base])[A[base] >= 0.1]), 1) for m in models if m != base})
print("by monkey, reading (a):", {mk: {label(m): round(100 * (A[m][monkey == mk].mean() / A[base][monkey == mk].mean() - 1), 1)
                                     for m in models if m != base} for mk in np.unique(monkey)})

### fig2b-contrast-normalization-no-effect

> Luminance and contrast normalization of the stimulus does not significantly change prediction
> accuracy compared with the simplest model. (Claim of absence: reproduced when the
> Bonferroni-corrected p over 11 comparisons is not below 0.05.)

In [ ]:
m = (True, 1.0, False)
pa, pb, pc, p = change(m)
print(f"luminance and contrast normalization, linear, no divisive: mean r {A[m].mean():.3f} against {A[base].mean():.3f}")
print(f"change: (a) {pa:.1f}%  (b) {pb:.1f}%  (c) {pc:.1f}%; better in {(A[m] > A[base]).sum()} of {len(IDX)}")
print(f"Wilcoxon p = {p:.4g}; Bonferroni x11 = {min(1, 11 * p):.4g}; x5 (variant) = {min(1, 5 * p):.4g}")
print(f"motion-enhanced validation only: p = {wilcoxon(A_enh[m], A_enh[base]).pvalue:.4g}")

**Result: reproduced.** Luminance and contrast normalization changes the mean correlation from 0.413 to 0.412 (reading (a) -0.3%, (b) -1.4%, (c) 0.6%; better in 26 of 45). Wilcoxon p = 0.414, Bonferroni-corrected p = 1 over 11 and over 5 comparisons, so the paper's test finds no effect.

### fig2b-compressive-improves

> A compressive static output nonlinearity consistently increases predictions over the simplest
> model, by about 16% without and about 19% with divisive normalization (each +- 4 points,
> corrected p < 0.01).

In [ ]:
def report(ms, expect):
    for m, e in zip(ms, expect):
        pa, pb, pc, p = change(m)
        print(f"{label(m)}: (a) {pa:.1f}%  (b) {pb:.1f}%  (c) {pc:.1f}%  [expected {e} +- 4]; "
              f"better in {(A[m] > A[base]).sum()} of {len(IDX)}; p = {p:.3g}, x11 = {min(1, 11 * p):.3g}, x5 = {min(1, 5 * p):.3g}")
        for rdg, v in zip("abc", (pa, pb, pc)):
            print(f"   reading ({rdg}) within tolerance: {abs(v - e) <= 4}")
report([(False, 0.5, False), (False, 0.5, True)], [16, 19])

**Result: partially reproduced.** The compressive nonlinearity improves predictions without and with divisive normalization in 40 of 45 neurons each, with corrected p below 1e-7 in both cases. The sizes are smaller than read from Figure 2B: 4.6% and 8.4% under reading (a), 9.2% and 16.1% under reading (b), 4.8% and 9.4% under reading (c), against 16 and 19 (+-4). Only reading (b) with divisive normalization is within tolerance, so no reading puts both values within it.

### fig2b-expansive-decreases

> An expansive static output nonlinearity always decreases predictions relative to the simplest
> model, by about 36% without and about 24% with divisive normalization.

In [ ]:
report([(False, 2.0, False), (False, 2.0, True)], [-36, -24])

**Result: partially reproduced.** The expansive nonlinearity lowers predictions without divisive normalization in all 45 neurons and with it in 38 of 45 (corrected p below 1e-7 for both). The sizes are -15.1% and -13.5% under reading (a), -26.9% and -24.5% under reading (b), -15.7% and -13.2% under reading (c), against -36 and -24 (+-4). Reading (b) matches the value with divisive normalization but not the one without it.

### fig2b-divisive-improves

> Divisive normalization significantly improves predictions over the simplest model, by about 8%
> with the linear static nonlinearity.

In [ ]:
report([(False, 1.0, True)], [8])

**Result: not reproduced.** Divisive normalization with the linear nonlinearity raises the mean correlation by 2.9% (a), 5.1% (b) or 3.9% (c); reading (b) is within 8 +- 4. It is better in 30 of 45 neurons, Wilcoxon p = 0.0051, but the Bonferroni-corrected p is 0.056 over 11 comparisons (0.026 over 5), so the claim's test (corrected p < 0.01) is not met.

### fig2b-compressive-divisive-best

> The model with both a compressive nonlinearity and divisive normalization predicts significantly
> better than the model with the compressive nonlinearity alone, and is the simplest model that
> gives accurate predictions.

In [ ]:
cd, c = (False, 0.5, True), (False, 0.5, False)
p = wilcoxon(A[cd], A[c]).pvalue
print(f"compressive + divisive {A[cd].mean():.3f} against compressive {A[c].mean():.3f}; better in {(A[cd] > A[c]).sum()} of {len(IDX)}")
print(f"Wilcoxon p = {p:.4g}; x11 = {min(1, 11 * p):.4g}; x5 = {min(1, 5 * p):.4g}; x12 = {min(1, 12 * p):.4g}")
print("models against compressive + divisive (positive = better than it):")
for m in models:
    if m == cd:
        continue
    pm = wilcoxon(A[m], A[cd]).pvalue
    print(f"  {label(m):28s} mean diff {A[m].mean() - A[cd].mean():+.4f}  p = {pm:.3g}  x11 = {min(1, 11 * pm):.3g}  x12 = {min(1, 12 * pm):.3g}")

**Result: reproduced.** The compressive and divisive model (mean r 0.448) beats the compressive model (0.432) in 34 of 45 neurons, Wilcoxon p = 0.00016, corrected p = 0.0017 over 11, 0.0019 over 12 and 0.0008 over 5. No other model is better on average; the closest, the same model with luminance normalization, is lower by 0.0065 (p = 0.36).

### fig2b-no-interaction

> The effects of the compressive and expansive nonlinearities do not depend on whether divisive
> normalization is present.

In [ ]:
for div in (False, True):
    for rdg, ref in (("a", base), ("b", (False, 1.0, div))):
        for a_ in (0.5, 2.0):
            m = (False, a_, div)
            d = A[m] - A[ref]
            p = wilcoxon(A[m], A[ref]).pvalue
            print(f"divisive {div!s:5s} reading ({rdg}) {label(m):22s} against {label(ref):14s}: mean diff {d.mean():+.4f}, "
                  f"p = {p:.3g}, x11 = {min(1, 11 * p):.3g}")

**Result: reproduced.** Without divisive normalization the compressive model is better than the linear one (+0.019, corrected p = 8.8e-8) and the expansive one worse (-0.063, 6.3e-13). With divisive normalization the same holds against the simplest model (reading (a): +0.035, p = 1.8e-8; -0.056, p = 4.5e-8) and against the linear divisive model (reading (b): +0.022, p = 8.1e-9; -0.068, p = 6.3e-13).

## complex-cells-dominate

> Most of the response variance is captured by the model complex cells, and the model simple
> cells have only a small effect on prediction performance. (Reproduced when the complex-only
> model reaches at least half of the full model's mean squared correlation and drops the mean
> correlation less than the simple-only model does.)

In [ ]:
full = A[mt2.MAIN]
co = pd.Series([acc(R[n]["fits"][mt2.MAIN]["complex_only"]["r"]) for n in names_run], index=IDX)
so = pd.Series([acc(R[n]["fits"][mt2.MAIN]["simple_only"]["r"]) for n in names_run], index=IDX)
print(f"mean r: full {full.mean():.3f}, complex only {co.mean():.3f}, simple only {so.mean():.3f}")
print(f"mean r^2: full {(full**2).mean():.4f}, complex only {(co**2).mean():.4f} "
      f"(ratio {(co**2).mean() / (full**2).mean():.3f}), simple only {(so**2).mean():.4f} (ratio {(so**2).mean() / (full**2).mean():.3f})")
print(f"drop of mean r: complex only {full.mean() - co.mean():.4f}, simple only {full.mean() - so.mean():.4f}")
print(f"Wilcoxon full vs complex only p = {wilcoxon(full, co).pvalue:.3g}; full vs simple only p = {wilcoxon(full, so).pvalue:.3g}")
print("regressors: complex", int((R[names_run[0]]["rd"][:, 5] == 0).sum()), ", simple", int((R[names_run[0]]["rd"][:, 5] == 1).sum()))

**Result: reproduced.** With complex-cell regressors only, the mean correlation is 0.448, the same as the full model (mean r squared ratio 1.001; Wilcoxon p = 0.054); with simple-cell regressors only it falls to 0.194 (ratio 0.250, p = 5.7e-14). Here the simple cells are limited to 0.75 cycles per cRF and SF 0 (218 of 763 regressors).

## Spectral receptive fields: the optimal plane

The optimal plane, on-plane ratio and horizontal-vertical ratio are computed as in `mt2.py`, under
the claims file's readings: balance of the two constraints (a) coverage with symmetry breaking
ties, (b) both scaled and summed; quantity (a) fitted weights, (b) weights times spectral
amplitude; sign split (a) per weight at each delay, (b) after summing over delays.

In [ ]:
rd = R[names_run[0]]["rd"]
amp = mt2.regressor_amplitude(rd)
near, side, PTH, PV = mt2.plane_geometry(rd)
RCOMB = [(b, q, s) for b in "ab" for q in "ab" for s in "ab"]

def plane_stats(w, sd):
    out = {}
    for b, q, s in RCOMB:
        E, I = mt2.split_weights(w, sd, amp, quantity=q, sign=s)
        j = mt2.optimal_plane(E, near, side, balance=b)
        out[(b, q, s)] = dict(exc=mt2.on_plane_ratio(E, near[:, j]), sup=mt2.on_plane_ratio(I, near[:, j]),
                              th=PTH[j], v=PV[j], hv_a=mt2.hv_ratio(E, rd, PTH[j], near[:, j], "a"),
                              hv_b=mt2.hv_ratio(E, rd, PTH[j], near[:, j], "b"))
    return out

real = {n: plane_stats(R[n]["fits"][mt2.MAIN]["w"], R[n]["fits"][mt2.MAIN]["sd"]) for n in names_run}
nulls = {}
for n in names_run:
    sdn = R[n]["fits"][mt2.MAIN]["sd"]
    for nl in R[n].get("nulls", []):
        key = (nl["id"][0], nl["rate"], nl["delay"])
        nulls.setdefault(key, []).append(plane_stats(nl["w"], sdn))
print({k: len(v) for k, v in nulls.items()})

### fig6a-null-on-plane

> For 150 model MT neurons with random weights, the average excitatory on-plane ratio is 0.36
> (expected 0.36 +- 0.036).

In [ ]:
rows = []
for key, lst in sorted(nulls.items()):
    for rc in RCOMB:
        v = np.array([x[rc]["exc"] for x in lst])
        rows.append(dict(movies=key[0], rate=key[1], delays=key[2], balance=rc[0], quantity=rc[1], sign=rc[2],
                         n=len(v), mean_exc=v.mean(), sd=v.std(), within=0.324 <= v.mean() <= 0.396))
NT = pd.DataFrame(rows)
print(NT.round(3).to_string())
print("\ncombinations within 0.324 to 0.396:", int(NT.within.sum()), "of", len(NT))

**Result: reproduced.** The mean excitatory on-plane ratio of the 150 null model neurons is within 0.324 to 0.396 in 12 of 40 combinations of readings. With every reading (a) it is 0.396 (SD 0.072), at the upper edge; null neurons made non-negative by an offset (rate reading (b)) give 0.44 to 0.53, outside the tolerance in every combination.

### fig6a-excitatory-on-plane

> Excitatory spectral receptive fields of MT neurons tend to lie on a single plane; their average
> on-plane ratio is 0.59 (+- 0.059), significantly above the null model's 0.36 (rank-sum p < 0.01).

In [ ]:
rows = []
for rc in RCOMB:
    v = np.array([real[n][rc]["exc"] for n in names_run])
    for key, lst in sorted(nulls.items()):
        nv = np.array([x[rc]["exc"] for x in lst])
        st = ranksums(v, nv)
        rows.append(dict(balance=rc[0], quantity=rc[1], sign=rc[2], mean_exc=v.mean(), null=f"{key[0]}{key[1]}{key[2]}",
                         null_mean=nv.mean(), p=st.pvalue, higher=v.mean() > nv.mean(),
                         ok=(0.531 <= v.mean() <= 0.649) and st.pvalue < 0.01 and v.mean() > nv.mean()))
ET = pd.DataFrame(rows)
print(ET.round(4).to_string())
print("\ncombinations meeting the criterion:", int(ET.ok.sum()), "of", len(ET))
for mk in np.unique(monkey):
    print(f"monkey {mk}, reading aaa: mean {np.mean([real[n][('a','a','a')]['exc'] for n, mm in zip(names_run, monkey) if mm == mk]):.3f}")

**Result: reproduced.** The mean excitatory on-plane ratio is 0.550 to 0.596 across the eight readings, within 0.531 to 0.649 in all of them, and above every null set by a rank-sum test (p from below 1e-4 to 0.0098), so all 40 combinations meet the criterion. With every reading (a) it is 0.592; monkey f 0.555, monkey v 0.705.

### fig6b-suppressive-off-plane

> Suppressive spectral receptive fields tend to avoid the optimal excitatory plane; their average
> on-plane ratio is 0.18 (+- 0.018), significantly below the null model (the null model's
> suppressive ratios; its excitatory ratios as a variant).

In [ ]:
rows = []
for rc in RCOMB:
    v = np.array([real[n][rc]["sup"] for n in names_run])
    for key, lst in sorted(nulls.items()):
        ns_ = np.array([x[rc]["sup"] for x in lst])
        ne_ = np.array([x[rc]["exc"] for x in lst])
        p1, p2 = ranksums(v, ns_).pvalue, ranksums(v, ne_).pvalue
        rows.append(dict(balance=rc[0], quantity=rc[1], sign=rc[2], mean_sup=v.mean(), null=f"{key[0]}{key[1]}{key[2]}",
                         null_sup=ns_.mean(), p_vs_null_sup=p1, null_exc=ne_.mean(), p_vs_null_exc=p2,
                         ok=(0.162 <= v.mean() <= 0.198) and p1 < 0.01 and v.mean() < ns_.mean()))
ST = pd.DataFrame(rows)
print(ST.round(4).to_string())
print("\ncombinations meeting the criterion:", int(ST.ok.sum()), "of", len(ST))

**Result: reproduced.** With the fitted weights split per delay (quantity (a), sign (a)), the mean suppressive on-plane ratio is 0.168 (balance (a)) or 0.171 (balance (b)), within 0.162 to 0.198, and below the null model's suppressive ratios of 0.196 to 0.209 for the null sets with rectified rates (rank-sum p 0.0008 to 0.0097). Six of 40 combinations meet the criterion. With the other readings the ratio is 0.094 to 0.159, below the tolerance band, and null sets made non-negative by an offset have suppressive ratios of 0.11 to 0.17 that do not differ from the real ones.

### fig7a-partial-ring

> The horizontal-vertical ratio is distributed continuously; few neurons form a full ring or a
> single peak, and most form a partial ring. (Reproduced when, under one combination, more than
> half of the neurons fall between the extremes and fewer than a quarter fall in each extreme;
> extremes (a) < 0.2 and >= 0.8, (b) < 0.1 and >= 0.9.)

In [ ]:
rows = []
HV = {}
for rc in RCOMB:
    for proj in ("a", "b"):
        v = np.array([real[n][rc][f"hv_{proj}"] for n in names_run])
        HV[(rc, proj)] = v
        for ext, (lo, hi) in (("a", (0.2, 0.8)), ("b", (0.1, 0.9))):
            fl, fr = np.mean(v < lo), np.mean(v >= hi)
            mid = 1 - fl - fr
            rows.append(dict(balance=rc[0], quantity=rc[1], sign=rc[2], amplitudes=proj, extremes=ext,
                             median=np.median(v), low=fl, ring=fr, between=mid,
                             ok=mid > 0.5 and fl < 0.25 and fr < 0.25))
HT = pd.DataFrame(rows)
print(HT.round(3).to_string())
print("\ncombinations meeting the criterion:", int(HT.ok.sum()), "of", len(HT))
v = HV[(("a", "a", "a"), "a")]
print("ratios, reading aaa/a: range", np.round(v.min(), 3), "to", np.round(v.max(), 3), "; infinite:", int(np.isinf(v).sum()))
fig, ax = plt.subplots(figsize=(4, 3))
ax.hist(np.clip(v, 0, 3), bins=30, color="0.4"); ax.set_xlabel("horizontal-vertical ratio (clipped at 3)")
print("points above 3 (clipped):", int((v > 3).sum()))

**Result: reproduced.** In 25 of 32 combinations more than half of the neurons fall between the extremes and fewer than a quarter in each. With every reading (a) and extremes (a), 6.7% of neurons have ratios below 0.2, 22.2% at or above 0.8, and 71.1% between them (median 0.534, range 0.055 to 1.652).

### fig7a-no-corr-prediction-hv

> Prediction performance is not significantly correlated with the horizontal-vertical ratio.
> (Pearson correlation, t test; reproduced when p > 0.10, not reproduced when p < 0.05.)

In [ ]:
pr_ = A[mt2.MAIN].values
rows = []
for (rc, proj), v in HV.items():
    ok = np.isfinite(v)
    r, p = pearsonr(pr_[ok], v[ok])
    rows.append(dict(balance=rc[0], quantity=rc[1], sign=rc[2], amplitudes=proj, n=int(ok.sum()), r=r, p=p))
CT = pd.DataFrame(rows)
print(CT.round(4).to_string())
print("\np > 0.10:", int((CT.p > 0.10).sum()), "; p < 0.05:", int((CT.p < 0.05).sum()), "of", len(CT))

**Result: partially reproduced.** With every reading (a), the correlation of prediction accuracy with the horizontal-vertical ratio is r = -0.281, p = 0.061, between the claim's bounds of 0.05 and 0.10. Across the 16 readings p > 0.10 in 10 and p < 0.05 in 2 (r from -0.380 to -0.120); every coefficient is negative.

### fig7b-pattern-index

> Simulated plaid responses give pattern indices that form a continuum, mostly from -6 to 2,
> with highly pattern-selective neurons rare. (Reproduced when more than half of the neurons are
> within -6.6 to 2.2 and fewer than 10% exceed 2.2.)

In [ ]:
def peak_pair(n):
    E, _ = mt2.split_weights(R[n]["fits"][mt2.MAIN]["w"])
    sel = (rd[:, 5] == 0) & (rd[:, 0] > 0) & (rd[:, 2] > 0)
    j = np.where(sel)[0][np.argmax(E[sel])]
    return (rd[j, 0], rd[j, 2])
pk = {n: peak_pair(n) for n in names_run}
vals, cnt = np.unique([f"{a}|{b}" for a, b in pk.values()], return_counts=True)
pop = tuple(float(x) for x in vals[np.argmax(cnt)].split("|"))
print("peak (SF cycles/cRF, TF Hz) counts:", dict(zip(vals, cnt)), "; population pair:", pop)
PI = {}
for fr in ("a", "b"):
    for sep in (120, 90):
        PI[(fr, sep)] = np.array([mt2.pattern_index(R[n]["fits"][mt2.MAIN]["plaid"][pk[n] if fr == "a" else pop]["grating"],
                                                    R[n]["fits"][mt2.MAIN]["plaid"][pk[n] if fr == "a" else pop][f"plaid{sep}"], sep)
                                  for n in names_run])
rows = []
for k, v in PI.items():
    rows.append(dict(frequency=k[0], separation=k[1], median=np.median(v), min=v.min(), max=v.max(),
                     in_range=np.mean((v >= -6.6) & (v <= 2.2)), above_2_2=np.mean(v > 2.2), above_1_28=np.mean(v > 1.28),
                     ok=np.mean((v >= -6.6) & (v <= 2.2)) > 0.5 and np.mean(v > 2.2) < 0.10))
PT = pd.DataFrame(rows)
print(PT.round(3).to_string())
print("\ncombinations meeting the criterion:", int(PT.ok.sum()), "of", len(PT))

**Result: not reproduced.** The simulated pattern indices are strongly component-like under all four combinations: medians -8.6 to -10.7, ranges within -14.6 to -4.4, so 0% to 18% of neurons lie within -6.6 to 2.2, against more than half required. No neuron has an index above 1.28. The grating and plaid parameters are this attempt's choice, as the paper gives none.

### fig7c-hv-pattern-correlation

> The horizontal-vertical ratio is significantly correlated with the simulated pattern index
> (r = 0.46 +- 0.046, p < 0.01).

In [ ]:
rows = []
for (rc, proj), hv in HV.items():
    for k, piv in PI.items():
        ok = np.isfinite(hv)
        r, p = pearsonr(hv[ok], piv[ok])
        rows.append(dict(balance=rc[0], quantity=rc[1], sign=rc[2], amplitudes=proj, frequency=k[0], separation=k[1],
                         r=r, p=p, ok=(0.414 <= r <= 0.506) and p < 0.01))
CC = pd.DataFrame(rows)
print(CC.round(4).to_string())
print("\ncombinations meeting the criterion:", int(CC.ok.sum()), "of", len(CC), "; r range", CC.r.min().round(3), "to", CC.r.max().round(3),
      "; positive with p < 0.01:", int(((CC.r > 0) & (CC.p < 0.01)).sum()))

**Result: not reproduced.** Across the 64 combinations of readings the correlation of the horizontal-vertical ratio with the pattern index ranges from -0.343 to 0.330, and none is positive with p < 0.01. This test rests on the pattern indices of fig7b-pattern-index, which do not match the paper's.

### fig7c-no-corr-prediction-pattern

> Prediction performance is not significantly correlated with the pattern index.

In [ ]:
rows = []
for k, piv in PI.items():
    r, p = pearsonr(pr_, piv)
    rows.append(dict(frequency=k[0], separation=k[1], r=r, p=p))
print(pd.DataFrame(rows).round(4).to_string())

**Result: reproduced.** With the per-neuron peak frequency and 120 degree plaids (the first readings), r = -0.231, p = 0.127, above 0.10. The other combinations give p = 0.902, 0.370 and 0.055. The pattern indices themselves do not match the paper's (fig7b-pattern-index).

## natural-movie-control

> Models fit on motion-enhanced movies predict responses to natural movies without motion
> enhancement as well as responses to new motion-enhanced movies (mean r = 0.533 against 0.537,
> 15 neurons; Wilcoxon signed-rank p > 0.10).

In [ ]:
nn = [n for n in names_run if "natural" in R[n]["fits"][mt2.MAIN]["r"]]
rn = np.array([R[n]["fits"][mt2.MAIN]["r"]["natural"] for n in nn])
re_ = np.array([R[n]["fits"][mt2.MAIN]["r"]["enhanced"] for n in nn])
st = wilcoxon(rn, re_)
print(f"{len(nn)} neurons with both validation sets ({', '.join(R[n]['cellid'] for n in nn)})")
print(f"mean r natural {rn.mean():.3f} [expected 0.533 +- 0.053: {abs(rn.mean() - 0.533) <= 0.053}], "
      f"motion-enhanced {re_.mean():.3f} [expected 0.537 +- 0.054: {abs(re_.mean() - 0.537) <= 0.054}]")
print(f"Wilcoxon signed-rank p = {st.pvalue:.4g}; natural higher in {(rn > re_).sum()} of {len(nn)}; mean difference {np.mean(rn - re_):+.3f}")
for n, a_, b_ in zip(nn, rn, re_):
    print(f"  {n[:6]}  natural {a_:.3f}  enhanced {b_:.3f}")

**Result: partially reproduced.** The files hold the natural-movie validation set for 14 neurons, not 15. The Wilcoxon signed-rank test finds no difference (p = 0.241; natural higher in 10 of 14, mean difference +0.031). The natural-movie mean, 0.493, is within 0.533 +- 0.053, but the motion-enhanced mean, 0.462, is below 0.537 +- 0.054.

## fig8-recovery-simulated

> The estimation procedure correctly recovers the spectral receptive fields of three simulated MT
> neurons. (Reproduced when each recovered spectral receptive field correlates better with its
> own true one than with the other two.)

In [ ]:
sims = R[SIM]["sims"] if SIM in R else []
def spec(w):
    E, _ = mt2.split_weights(w)
    keys = sorted({(r_[0], r_[1], r_[2]) for r_ in rd[rd[:, 5] == 0]})
    idx = {k: i for i, k in enumerate(keys)}
    out = np.zeros(len(keys))
    for j in np.where(rd[:, 5] == 0)[0]:
        out[idx[(rd[j, 0], rd[j, 1], rd[j, 2])]] += E[j]
    return out
for noise in ("a", "b"):
    S_ = [s for s in sims if s["noise"] == noise]
    M = np.array([[np.corrcoef(spec(a["w"]), spec(b["w_true"]))[0, 1] for b in S_] for a in S_])
    print(f"noise reading ({noise}): rows recovered A, B, C; columns true A, B, C")
    print(np.round(M, 3))
    print("  each recovered field closest to its own:", bool(np.all(np.argmax(M, 1) == np.arange(3))))

**Result: reproduced.** For each of the three simulated neurons the recovered spectral receptive field correlates best with its own true field: with Poisson noise 0.835, 0.570 and 0.501 on the diagonal against at most 0.133 off it, and without noise 0.791, 0.600 and 0.672 against at most 0.072.

## Summary

| Claim | Outcome |
|---|---|
| fig3b-mean-prediction | not reproduced (0.448 against 0.52 +- 0.052) |
| fig3b-explainable-variance | blocked: data_missing |
| fig3a-example-neuron | reproduced |
| fig2b-contrast-normalization-no-effect | reproduced |
| fig2b-compressive-improves | partially reproduced (significant, smaller) |
| fig2b-expansive-decreases | partially reproduced (significant, one size outside) |
| fig2b-divisive-improves | not reproduced (corrected p = 0.056) |
| fig2b-compressive-divisive-best | reproduced |
| fig2b-no-interaction | reproduced |
| complex-cells-dominate | reproduced |
| fig6a-null-on-plane | reproduced (12 of 40 combinations) |
| fig6a-excitatory-on-plane | reproduced |
| fig6b-suppressive-off-plane | reproduced (6 of 40 combinations) |
| fig7a-partial-ring | reproduced |
| fig7a-no-corr-prediction-hv | partially reproduced (p = 0.061) |
| fig7b-pattern-index | not reproduced |
| fig7c-hv-pattern-correlation | not reproduced |
| fig7c-no-corr-prediction-pattern | reproduced |
| natural-movie-control | partially reproduced |
| fig8-recovery-simulated | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The regression weights are constrained to be pooled over space (763 regressors at 11 delays),
  where the paper fits each of its 5956 filters; the filter bank itself follows the paper's rules
  (about 7000 complex and 1100 simple channels). Simple cells enter the regression only at 0.75
  cycles per cRF and SF 0.
- The paper does not give the temporal envelope of its Gabor filters. Here it spans 11 frames
  (SD 2 frames), centred on the frame, and the 11-frame delay window is moved 0, 3 or 6 frames
  earlier per neuron, chosen on held-out training data (0 for 31 neurons).
- Five temporal frequencies (0 to 30 Hz) and five spatial frequencies (0 to 6 cycles per cRF), an
  octave apart; sigma of divisive normalization 1% of the mean pooled response (and chosen on
  held-out data as a reading); boosting step 1% of the response SD; early stopping after 50 steps
  without improvement.
- 45 neurons where the paper has 52, and 14 with a natural-movie validation set where it has 15.
- Plaid and grating parameters, the plane search grid and the quadrant geometry of the
  horizontal-vertical ratio are this attempt's choices; the paper gives none.

What was found about the dataset:

- 45 of the paper's 52 neurons (34 from monkey f, 11 from monkey v); no recording appears twice.
- The files hold only responses averaged over repeats, so the explainable variance cannot be
  computed.
- 14 files have an undocumented `ranges.nrange`, the first 2000 frames of a 4000-frame `vrange`;
  its frames carry less motion than the rest of `vrange` in all 14 (0.225 against 0.325, median
  frame difference over contrast), consistent with the natural movies of the paper's control.
  `ranges.trange` is empty in every file and not described.
- The per-frame denominators of the averaged validation responses imply 10 to 33 repeats, where
  the paper reports 5 to 20; training frames carry denominators up to 5.
- Training frames run from 17,908 to 40,000 per neuron (mean 26,894; the paper gives 20,000 to
  40,000, mean 27,120). In ct0133, 12,729 of 19,842 training frames have no response.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min")